In [ ]:
import pandas as pd

df = pd.read_csv("/Users/jeremycheng/Documents/yhack_2026/data/us_county_latlng.csv")

In [ ]:
df.head()

In [ ]:
state_map = {
    "01": "Alabama",
    "02": "Alaska",
    "04": "Arizona",
    "05": "Arkansas",
    "06": "California",
    "08": "Colorado",
    "09": "Connecticut",
    "10": "Delaware",
    "11": "District of Columbia",
    "12": "Florida",
    "13": "Georgia",
    "15": "Hawaii",
    "16": "Idaho",
    "17": "Illinois",
    "18": "Indiana",
    "19": "Iowa",
    "20": "Kansas",
    "21": "Kentucky",
    "22": "Louisiana",
    "23": "Maine",
    "24": "Maryland",
    "25": "Massachusetts",
    "26": "Michigan",
    "27": "Minnesota",
    "28": "Mississippi",
    "29": "Missouri",
    "30": "Montana",
    "31": "Nebraska",
    "32": "Nevada",
    "33": "New Hampshire",
    "34": "New Jersey",
    "35": "New Mexico",
    "36": "New York",
    "37": "North Carolina",
    "38": "North Dakota",
    "39": "Ohio",
    "40": "Oklahoma",
    "41": "Oregon",
    "42": "Pennsylvania",
    "44": "Rhode Island",
    "45": "South Carolina",
    "46": "South Dakota",
    "47": "Tennessee",
    "48": "Texas",
    "49": "Utah",
    "50": "Vermont",
    "51": "Virginia",
    "53": "Washington",
    "54": "West Virginia",
    "55": "Wisconsin",
    "56": "Wyoming",
    "60": "American Samoa",
    "66": "Guam",
    "69": "Northern Mariana Islands",
    "72": "Puerto Rico",
    "78": "U.S. Virgin Islands"
}

In [ ]:
df['state_fips'] = df['fips_code'].astype(str).str.zfill(5).str[:2]
df['state'] = df['state_fips'].map(state_map)

In [ ]:
df

In [ ]:
# Only keep 50 states
df = df[df['state_fips'].astype(int) <= 56]

In [ ]:
df.shape

In [ ]:
# NaN check
df.isna().sum().sum()

In [ ]:
# df.to_csv("us_county_clean.csv", index=False)

In [ ]:
df = df.rename(columns={"lng": "lon"})

In [ ]:
df.head()

In [ ]:
import time
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

DAILY_VARS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "rain_sum",
    "uv_index_max",
    "snowfall_sum",
    "precipitation_hours",
]

In [ ]:
def fetch_county_weather(counties_df, batch_size=50):
    """
    Fetch daily weather forecasts from Open-Meteo for every county, summarize
    each into a single row of annual-style features, and return the merged
    DataFrame.

    Args:
        counties_df: DataFrame with at least [fips_code, name, lat, lon].
        batch_size: Number of locations per API call.

    Returns:
        A copy of counties_df with weather summary columns appended.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    lats = counties_df["lat"].tolist()
    lons = counties_df["lon"].tolist()
    summaries = []

    for start in range(0, len(lats), batch_size):
        end = min(start + batch_size, len(lats))
        params = {
            "latitude": lats[start:end],
            "longitude": lons[start:end],
            "daily": DAILY_VARS,
            "models": "gfs_seamless",
            "timezone": "GMT",
        }

        responses = openmeteo.weather_api(url, params=params)

        for response in responses:
            daily = response.Daily()
            dates = pd.date_range(
                start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=daily.Interval()),
                inclusive="left",
            )
            day_df = pd.DataFrame({"date": dates})
            for i, var in enumerate(DAILY_VARS):
                day_df[var] = daily.Variables(i).ValuesAsNumpy()

            summaries.append({
                "avg_temp_max": day_df["temperature_2m_max"].mean(),
                "avg_temp_min": day_df["temperature_2m_min"].mean(),
                "avg_rain": day_df["rain_sum"].mean(),
                "total_rain": day_df["rain_sum"].sum(),
                "avg_uv_index": day_df["uv_index_max"].mean(),
                "max_uv_index": day_df["uv_index_max"].max(),
                "total_snowfall": day_df["snowfall_sum"].sum(),
                "avg_precip_hours": day_df["precipitation_hours"].mean(),
                "total_precip_hours": day_df["precipitation_hours"].sum(),
                "forecast_days": len(day_df),
            })

        print(f"  Processed {end}/{len(lats)} counties")
        if end % 550 == 0:
            print("  Approaching rate limit, waiting 65s...")
            time.sleep(65)

    result = counties_df.reset_index(drop=True).copy()
    weather_df = pd.DataFrame(summaries)
    return pd.concat([result, weather_df], axis=1)

In [ ]:
final_df = fetch_county_weather(df)
print(f"Shape: {final_df.shape}")
final_df.head(10)

In [ ]:
final_df.describe()

In [ ]:
# final_df.to_csv("data/us_county_weather.csv", index=False)

In [ ]:
final_df